In [13]:
import optuna.exceptions
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Load Data

In [15]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    transforms.Resize((256,256)),
    transforms.RandomResizedCrop(
        224,
        scale=(0.80, 1.0)
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [16]:
dataset_path = '../dataset'

dataset = datasets.ImageFolder(root=dataset_path, transform=image_transforms)
len(dataset)

2300

In [17]:
dataset.classes

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [18]:
num_classes = len(dataset.classes)
num_classes

6

In [19]:
train_size = int(0.75*len(dataset))
val_size = len(dataset) - train_size

train_size, val_size

(1725, 575)

In [20]:
from torch.utils.data import random_split

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [21]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, pin_memory=True, num_workers=2)

## Model 5: EfficientNet V2-S with Hyper parameter tuning

In [22]:
class CarClassifierEffNet(nn.Module):
    def __init__(self, num_classes, dropout_rate):
        super().__init__()
        self.model = models.efficientnet_v2_s(weights='DEFAULT')
        # Freeze all layers except the final fully connected layer
        for param in self.model.parameters():
            param.requires_grad = False

        # Retrieve the in_features of the original classifier (which is a Sequential block)
        in_features = self.model.classifier[1].in_features
        # Unfreeze top 2 layers and fc layers
        for param in self.model.features[-2:].parameters():
            param.requires_grad = True
        # Replace the final fully connected layer
        self.model.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [23]:
def objective(trial):
    # Suggest values for the hyperparameters
    lr_bb = trial.suggest_float('lr_bb', 1e-6, 1e-4, log=True)
    lr_cl = trial.suggest_float('lr_cl', 1e-4, 1e-3, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    # load the model
    model = CarClassifierEffNet(num_classes=num_classes, dropout_rate=dropout_rate).to(device)

    # define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW([
        {'params': model.model.features[-2:].parameters(), 'lr_bb': lr_bb},
        {'params': model.model.classifier.parameters(), 'lr_cl': lr_cl}
    ], weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.2)
    scaler = torch.amp.GradScaler('cuda')

    # training the loop
    epochs = 15
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_num, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)
            # Zero the parameter gradients
            optimizer.zero_grad()
            # Forward pass
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            # Backward pass
            scaler.scale(loss).backward()
            # Update the weights
            scaler.step(optimizer)
            scaler.update()

            if (batch_num+1) % 10 == 0:
                print(f"Batch: {batch_num+1}, Epoch: {epoch+1}, Loss: {loss.item():.2f}")

            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss/len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

    # Model Evaluation:

        model.eval()
        total = 0.0
        correct = 0.0
        val_loss = 0.0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_accuracy = 100 * correct / total

        trial.report(val_accuracy, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        # FIXED: Call the scheduler step at the end of the epoch using validation loss
        scheduler.step(avg_val_loss)

    return val_accuracy

In [24]:
import optuna

# 1. Add a pruner to kill bad hyperparameter trials early
# startup_trials=5: Allows the first 5 trials to run fully to establish a baseline performance median
# n_warmup_steps=3: Gives trials 3 epochs to warm up before they are eligible for pruning
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)

# 2. Run the optimization
study.optimize(objective, n_trials=20)

# 3. Print the breakthrough results
print("Best trial visual baseline accuracy:")
print(f"  Value: {study.best_trial.value:.2f}%")
print("  Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-11-04 18:53:04,279] A new study created in memory with name: no-name-f25b7f76-8405-4f92-b933-9fcb5eb050e8


Batch: 10, Epoch: 1, Loss: 1.68
Batch: 20, Epoch: 1, Loss: 1.48
Batch: 30, Epoch: 1, Loss: 1.36
Batch: 40, Epoch: 1, Loss: 1.21
Batch: 50, Epoch: 1, Loss: 1.16
Epoch [1/15], Loss: 1.4195
Batch: 10, Epoch: 2, Loss: 1.18
Batch: 20, Epoch: 2, Loss: 1.31
Batch: 30, Epoch: 2, Loss: 0.70
Batch: 40, Epoch: 2, Loss: 0.88
Batch: 50, Epoch: 2, Loss: 0.78
Epoch [2/15], Loss: 0.9929
Batch: 10, Epoch: 3, Loss: 0.80
Batch: 20, Epoch: 3, Loss: 0.78
Batch: 30, Epoch: 3, Loss: 1.10
Batch: 40, Epoch: 3, Loss: 0.78
Batch: 50, Epoch: 3, Loss: 0.89
Epoch [3/15], Loss: 0.8487
Batch: 10, Epoch: 4, Loss: 0.84
Batch: 20, Epoch: 4, Loss: 0.75
Batch: 30, Epoch: 4, Loss: 0.53
Batch: 40, Epoch: 4, Loss: 0.62
Batch: 50, Epoch: 4, Loss: 1.04
Epoch [4/15], Loss: 0.8116
Batch: 10, Epoch: 5, Loss: 0.53
Batch: 20, Epoch: 5, Loss: 0.89
Batch: 30, Epoch: 5, Loss: 0.83
Batch: 40, Epoch: 5, Loss: 0.70
Batch: 50, Epoch: 5, Loss: 0.85
Epoch [5/15], Loss: 0.7657
Batch: 10, Epoch: 6, Loss: 0.45
Batch: 20, Epoch: 6, Loss: 0.80
B

[I 2026-11-04 19:12:27,689] Trial 0 finished with value: 72.0 and parameters: {'lr_bb': 5.611516415334504e-06, 'lr_cl': 0.0008927180304353628, 'dropout_rate': 0.5659969709057024, 'weight_decay': 6.251373574521755e-05}. Best is trial 0 with value: 72.0.


Batch: 10, Epoch: 1, Loss: 1.77
Batch: 20, Epoch: 1, Loss: 1.64
Batch: 30, Epoch: 1, Loss: 1.70
Batch: 40, Epoch: 1, Loss: 1.64
Batch: 50, Epoch: 1, Loss: 1.51
Epoch [1/15], Loss: 1.6834
Batch: 10, Epoch: 2, Loss: 1.55
Batch: 20, Epoch: 2, Loss: 1.53
Batch: 30, Epoch: 2, Loss: 1.39
Batch: 40, Epoch: 2, Loss: 1.31
Batch: 50, Epoch: 2, Loss: 1.19
Epoch [2/15], Loss: 1.3776
Batch: 10, Epoch: 3, Loss: 1.42
Batch: 20, Epoch: 3, Loss: 1.13
Batch: 30, Epoch: 3, Loss: 1.02
Batch: 40, Epoch: 3, Loss: 1.05
Batch: 50, Epoch: 3, Loss: 1.29
Epoch [3/15], Loss: 1.1440
Batch: 10, Epoch: 4, Loss: 1.18
Batch: 20, Epoch: 4, Loss: 0.99
Batch: 30, Epoch: 4, Loss: 0.79
Batch: 40, Epoch: 4, Loss: 0.91
Batch: 50, Epoch: 4, Loss: 0.79
Epoch [4/15], Loss: 0.9903
Batch: 10, Epoch: 5, Loss: 0.90
Batch: 20, Epoch: 5, Loss: 0.95
Batch: 30, Epoch: 5, Loss: 0.83
Batch: 40, Epoch: 5, Loss: 0.99
Batch: 50, Epoch: 5, Loss: 0.97
Epoch [5/15], Loss: 0.9074
Batch: 10, Epoch: 6, Loss: 1.06
Batch: 20, Epoch: 6, Loss: 0.73
B

[I 2026-11-04 19:31:30,176] Trial 1 finished with value: 68.69565217391305 and parameters: {'lr_bb': 2.05133826308745e-06, 'lr_cl': 0.00014321698289111514, 'dropout_rate': 0.22904180608409974, 'weight_decay': 0.0003967605077052988}. Best is trial 0 with value: 72.0.


Batch: 10, Epoch: 1, Loss: 1.71
Batch: 20, Epoch: 1, Loss: 1.50
Batch: 30, Epoch: 1, Loss: 1.33
Batch: 40, Epoch: 1, Loss: 1.14
Batch: 50, Epoch: 1, Loss: 1.03
Epoch [1/15], Loss: 1.3901
Batch: 10, Epoch: 2, Loss: 0.95
Batch: 20, Epoch: 2, Loss: 0.77
Batch: 30, Epoch: 2, Loss: 0.76
Batch: 40, Epoch: 2, Loss: 0.84
Batch: 50, Epoch: 2, Loss: 0.89
Epoch [2/15], Loss: 0.8571
Batch: 10, Epoch: 3, Loss: 0.85
Batch: 20, Epoch: 3, Loss: 0.50
Batch: 30, Epoch: 3, Loss: 0.66
Batch: 40, Epoch: 3, Loss: 0.53
Batch: 50, Epoch: 3, Loss: 0.64
Epoch [3/15], Loss: 0.6989
Batch: 10, Epoch: 4, Loss: 0.54
Batch: 20, Epoch: 4, Loss: 0.56
Batch: 30, Epoch: 4, Loss: 0.45
Batch: 40, Epoch: 4, Loss: 0.63
Batch: 50, Epoch: 4, Loss: 0.59
Epoch [4/15], Loss: 0.6296
Batch: 10, Epoch: 5, Loss: 0.49
Batch: 20, Epoch: 5, Loss: 0.77
Batch: 30, Epoch: 5, Loss: 0.44
Batch: 40, Epoch: 5, Loss: 0.65
Batch: 50, Epoch: 5, Loss: 0.78
Epoch [5/15], Loss: 0.5711
Batch: 10, Epoch: 6, Loss: 0.67
Batch: 20, Epoch: 6, Loss: 0.57
B

... (18 outputs left out)

[I 2026-11-04 23:16:43,715] Trial 11 finished with value: 78.43478260869566 and parameters: {'lr_bb': 9.310048252433514e-05, 'lr_cl': 0.0002275489529463422, 'dropout_rate': 0.6850163016687525, 'weight_decay': 1.0811131532236843e-06}. Best is trial 3 with value: 78.95652173913044.


Batch: 10, Epoch: 1, Loss: 1.71
Batch: 20, Epoch: 1, Loss: 1.66
Batch: 30, Epoch: 1, Loss: 1.37


[W 2026-11-04 23:17:24,375] Trial 12 failed with parameters: {'lr_bb': 8.779088538192306e-05, 'lr_cl': 0.000208495220625197, 'dropout_rate': 0.4748577341268124, 'weight_decay': 1.0644430856079222e-06} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "F:\PyCharm 2025.1.2\projects\.venv\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Aniruddh_G\AppData\Local\Temp\ipykernel_30480\2130764623.py", line 35, in objective
    scaler.scale(loss).backward()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "F:\PyCharm 2025.1.2\projects\.venv\.venv\Lib\site-packages\torch\_tensor.py", line 623, in backward
    torch.autograd.backward(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        self, gradient, retain_graph, create_graph, inputs=inputs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "F:\PyCharm 2025.1.2\projects\.venv\.venv\Lib\site-packages\torch\autogr

KeyboardInterrupt: 

In [ ]:
study.best_params